In [1]:
from pathlib import Path
import re
import zipfile
import pandas as pd
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
import branca

In [2]:


# === CONFIGURAÇÃO ===
PASTA = Path(r"D:\Arq-Azzoni\UrbanSprawl\Bases_dados\Clima\Cobertura_vegetal")   # pasta onde estão os .zip
PADRAO_ANO = re.compile(r'(\d{4})$')# como extrair o ano do nome (ex.: dados_2019.zip)
NOME_CSV_INTERNO = None  # se cada ZIP tiver + de 1 arquivo e você souber o nome, coloque aqui (ex.: "tabela.csv")
SEP = ","                # separador do CSV (','; ';'; '\t', etc.)
ENCODING = "utf-8"       # encoding (ex.: "latin1" para acentuação antiga)
USECOLS = None           # opcional: lista de colunas para carregar
DTYPES = None            # opcional: dict de tipos por coluna, ex.: {"id":"string","valor":"float64"}
NA_VALUES = ["", "NA", "NaN", "null"]  # valores tratados como nulos

# === PROCESSAMENTO ===
tabelas = []

for zip_path in sorted(PASTA.glob("*.zip")):
    # extrair o ano do nome do arquivo ZIP
    m = PADRAO_ANO.search(zip_path.stem)
    if not m:
        print(f"Aviso: não encontrei ano no nome: {zip_path.name}. Pulando.")
        continue
    ano = int(m.group(1))

    with zipfile.ZipFile(zip_path) as zf:
        # escolha do arquivo CSV dentro do ZIP
        if NOME_CSV_INTERNO:
            member = NOME_CSV_INTERNO
        else:
            # supõe 1 CSV por ZIP; se houver vários, pega o primeiro .csv
            csv_members = [n for n in zf.namelist() if n.lower().endswith(".csv")]
            if not csv_members:
                print(f"Aviso: não há CSV em {zip_path.name}. Pulando.")
                continue
            if len(csv_members) > 1:
                print(f"Aviso: há {len(csv_members)} CSVs em {zip_path.name}. Usando o primeiro: {csv_members[0]}")
            member = csv_members[0]

        with zf.open(member) as f:
            df = pd.read_csv(
                f,
                sep=SEP,
                encoding=ENCODING,
                usecols=USECOLS,
                dtype=DTYPES,
                na_values=NA_VALUES,
                low_memory=False
            )

    # adiciona coluna de ano
    df["ano"] = ano

    tabelas.append(df)

# empilha tudo
if not tabelas:
    raise SystemExit("Nenhum dado carregado. Verifique a pasta e os nomes dos arquivos.")

final = pd.concat(tabelas, ignore_index=True)

# checagem rápida de consistência de colunas
# (opcional) garante mesma ordem de colunas (exceto 'ano' adicionado no final)
cols = [c for c in final.columns if c != "ano"] + ["ano"]
final = final[cols]

# === SAÍDA ===
# OUT_CSV = "tabela_todos_anos.csv"
# OUT_PARQUET = "tabela_todos_anos.parquet"

# final.to_csv(OUT_CSV, index=False)
# final.to_parquet(OUT_PARQUET, index=False)

# print(f"Linhas totais: {len(final):,}")
# print(f"Colunas: {len(final.columns)}")
# print(f"Salvo em: {OUT_CSV} e {OUT_PARQUET}")


In [3]:
# filtra só os anos de interesse
df_2016 = final[final["ano"] == 2016][["cd_setor", "pcv"]].rename(columns={"pcv": "pcv_2016"})
df_2023 = final[final["ano"] == 2023][["cd_setor", "pcv"]].rename(columns={"pcv": "pcv_2023"})

# junta os dois dataframes pelo cd_setor
df_diff = pd.merge(df_2023, df_2016, on="cd_setor", how="inner")


In [4]:
# garante numéricos (se já não estiver)
df_diff["pcv_2023"] = pd.to_numeric(df_diff["pcv_2023"], errors="coerce")
df_diff["pcv_2016"] = pd.to_numeric(df_diff["pcv_2016"], errors="coerce")

# cria as colunas
df_diff["pcv_diff"] = df_diff["pcv_2023"] - df_diff["pcv_2016"]


In [7]:
df_diff.to_csv("./data/dif_cobertura.csv", index=False)

In [7]:
df_diff = pd.read_csv("./data/dif_cobertura.csv")

In [5]:
df_diff['pcv_diff'].describe()

count    1085.000000
mean        2.412664
std         3.853272
min       -39.160000
25%         0.600000
50%         1.980000
75%         4.060000
max        19.960000
Name: pcv_diff, dtype: float64

In [9]:
df_diff

,cd_setor,pcv_2023,pcv_2016,pcv_diff
0,352590405000002,3.87,2.78,1.09
1,352590405000003,5.89,5.56,0.33
2,352590405000004,6.27,5.61,0.66
3,352590405000006,5.50,5.91,-0.41
4,352590405000007,6.36,5.47,0.89
...,...,...,...,...
1083,352590405001609,35.76,33.11,2.65
1084,352590405001611,11.53,12.66,-1.13
1085,352590405001612,11.10,10.38,0.72
1086,352590405001613,15.37,13.14,2.23


In [5]:


# ============================
# 1) ENTRADAS / CONFIGURAÇÃO
# ============================
CAMINHO_SHP = r"D:\Arq-Azzoni\UrbanSprawl\Bases_dados\Shapes\SP_setores_CD2022\SP_setores_CD2022.shp"  # << ajuste o caminho do seu shapefile
SHP_CHAVE = "CD_SETOR"                         # << ajuste o nome da coluna-chave no shapefile (ex.: "CD_SETOR", "CD_GEOCODI", etc.)
VALUE_COL = "pcv_diff"                      # pode usar "pcv_var_pct" (percentual) ou "pcv_diff" (absoluto)

# df_diff deve existir da etapa anterior e conter as colunas:
# ["cd_setor", "pcv_2016", "pcv_2023", "pcv_diff", "pcv_var_pct"]

# ============================
# 2) LEITURA E PREPARO DO SHP
# ============================
gdf = gpd.read_file(CAMINHO_SHP)

In [17]:


# chave como string
gdf["cd_setor"] = gdf[SHP_CHAVE].astype(str)
df_diff["cd_setor"] = df_diff["cd_setor"].astype(str)

# 🔹 mantém apenas setores presentes em df_diff
gdf = gdf[gdf["cd_setor"].isin(df_diff["cd_setor"])].copy()

# garante CRS adequado
gdf = gdf.set_crs(epsg=4326, allow_override=True) if gdf.crs is None else gdf.to_crs(epsg=4326)

# ===== MERGE =====
df_use = df_diff[["cd_setor", VALUE_COL]].copy()
df_use[VALUE_COL] = pd.to_numeric(df_use[VALUE_COL], errors="coerce")
gdf_m = gdf.merge(df_use, on="cd_setor", how="left")

# # limites simétricos
# vals = gdf_m[VALUE_COL].astype(float)
# v = np.nanmax(np.abs(vals.values))
# v = 1.0 if (not np.isfinite(v) or v == 0) else v
# vmin, vmax = -v, v

# # garante que o "miolo branco" não extrapole o alcance
# inner = min(2.0, vmax)


# cmap = branca.colormap.LinearColormap(
#     colors=["darkred", "white", "white", "white", "darkgreen"],
#     index=[vmin, -inner, 0, inner, vmax],
#     vmin=vmin, vmax=vmax
# ).to_step(9)

# # limites simétricos
# vals = gdf_m[VALUE_COL].astype(float)
# v = np.nanmax(np.abs(vals.values))
# v = 1.0 if (not np.isfinite(v) or v == 0) else v
# vmin, vmax = -v, v

# # faixa branca só até |dif| < 1
# inner = min(1.0, vmax)

# cmap = branca.colormap.LinearColormap(
#     colors=["darkred", "white", "white", "white", "darkgreen"],
#     index=[vmin, -inner, 0, inner, vmax],
#     vmin=vmin, vmax=vmax
# ).to_step(9)

# def style_fn(feature):
#     val = feature["properties"].get(VALUE_COL, None)
#     if val is None:
#         return {"fillOpacity": 0.2, "weight": 0.1, "color": "#999", "fillColor": "#cccccc"}
#     try:
#         color = cmap(float(val))
#     except Exception:
#         color = "#cccccc"
#     return {"fillColor": color, "color": "#555", "weight": 0.3, "fillOpacity": 0.8}

THRESH = 1.0  # branco se |dif| < 1

# ===== LIMITES ROBUSTOS (assimétricos e ancorados em 0) =====
vals = pd.to_numeric(gdf_m[VALUE_COL], errors="coerce").astype(float).to_numpy()
low, high = np.nanpercentile(vals, [2, 98])  # corta extremos
# garante que 0 fique dentro e que a faixa branca ±1 caiba:
low  = min(low, -THRESH)
high = max(high, THRESH)
vmin, vmax = float(low), float(high)

# ===== COLORMAP CONTÍNUO COM FAIXA CENTRAL BRANCA =====
# três “pivôs”: negativo forte -> branco -> positivo forte
inner = min(THRESH, vmax)
cmap = branca.colormap.LinearColormap(
    colors=["darkred", "white", "white", "white", "darkgreen"],
    index=[vmin, -inner, 0, inner, vmax],
    vmin=vmin, vmax=vmax
)
cmap.caption = "Escala (negativo → positivo)"

def style_fn(feature):
    val = feature["properties"].get(VALUE_COL, None)
    if val is None or pd.isna(val):
        return {"fillOpacity": 0.2, "weight": 0.1, "color": "#999", "fillColor": "#cccccc"}

    try:
        x = float(val)
    except Exception:
        return {"fillColor": "#cccccc", "color": "#555", "weight": 0.3, "fillOpacity": 0.8}

    # branco ESTRITAMENTE para |x| < 1
    if abs(x) < THRESH:
        color = "#ffffff"
    else:
        color = cmap(x)

    return {"fillColor": color, "color": "#555", "weight": 0.3, "fillOpacity": 0.8}

def highlight_fn(_):
    return {"weight": 1.5, "color": "#000", "fillOpacity": 0.9}

# ===== MAPA =====
bounds = gdf_m.total_bounds
centro = [(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2]
m = folium.Map(location=centro, zoom_start=12, tiles="cartodbpositron")

tooltip = folium.features.GeoJsonTooltip(
    fields=["cd_setor", VALUE_COL],
    aliases=["Setor:", "Valor:"],
    localize=True, sticky=True
)

folium.GeoJson(
    data=gdf_m,
    style_function=style_fn,
    highlight_function=highlight_fn,
    tooltip=tooltip,
    name="Camada"
).add_to(m)

cmap.caption = "Escala (negativo → positivo)"
cmap.add_to(m)

# remove números da escala (via CSS inject)
from branca.element import Element
hide_numbers = Element("""
<style>
.leaflet-container .branca-colorbar .tick text {
    display: none !important;
}
</style>
""")
m.get_root().html.add_child(hide_numbers)


m.save("mapa_setores.html")
print("Mapa salvo em: mapa_setores.html")

Mapa salvo em: mapa_setores.html


In [ ]:
m

In [23]:
df_diff.iloc[0]

cd_setor    352590405000002
pcv_2023               3.87
pcv_2016               2.78
Name: 0, dtype: object

In [ ]:

# opcional: ordena pelo maior aumento ou redução
df_diff = df_diff.sort_values("pcv_diff", ascending=False)

In [ ]:
import numpy as np

# (re)partindo do df_diff já criado no passo anterior
# df_diff tem colunas: cd_setor, pcv_2023, pcv_2016

# variação absoluta (já tínhamos)
df_diff["pcv_diff"] = df_diff["pcv_2023"] - df_diff["pcv_2016"]

# variação percentual: ((2023 - 2016) / 2016) * 100
# trata divisão por zero e faltantes
df_diff["pcv_var_pct"] = np.where(
    (df_diff["pcv_2016"].notna()) & (df_diff["pcv_2016"] != 0),
    (df_diff["pcv_diff"] / df_diff["pcv_2016"]) * 100,
    np.nan
)

# opcional: arredondar e criar versão formatada com %
df_diff["pcv_var_pct"] = df_diff["pcv_var_pct"].round(2)
df_diff["pcv_var_pct_str"] = df_diff["pcv_var_pct"].map(lambda x: f"{x:.2f}%" if pd.notna(x) else None)

# exemplo de ordenação: maior aumento percentual primeiro
df_diff = df_diff.sort_values("pcv_var_pct", ascending=False)

print(df_diff.head())

In [13]:
final

,cd_mun,nm_mun,cd_setor,pcv,ano
0,3525904,Jundiaí,352590405000002,2.78,2016
1,3525904,Jundiaí,352590405000003,5.56,2016
2,3525904,Jundiaí,352590405000004,5.61,2016
3,3525904,Jundiaí,352590405000006,5.91,2016
4,3525904,Jundiaí,352590405000007,5.47,2016
...,...,...,...,...,...
8699,3525904,Jundiaí,352590405001609,35.76,2023
8700,3525904,Jundiaí,352590405001611,11.53,2023
8701,3525904,Jundiaí,352590405001612,11.1,2023
8702,3525904,Jundiaí,352590405001613,15.37,2023


In [ ]:
#keep setores do municipio jundiaí